<a href="https://colab.research.google.com/github/aryasure/BA305-UK-Mapping-Musem-Project/blob/main/Museums_RF_Predictive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier

In [2]:
df = pd.read_csv("museums_cleaned.csv")
target = "size"
df[target] = df[target].astype("string").str.strip().str.lower()
df = df[df[target].notna() & (df[target] != "")].copy()

# Convert year_opened values like "1984:1984" into a numeric midpoint.
if "year_opened" in df.columns:
    parts = df["year_opened"].astype("string").str.extract(r"^\s*(\d{4})\s*:\s*(\d{4})\s*$")
    start, end = pd.to_numeric(parts[0], errors="coerce"), pd.to_numeric(parts[1], errors="coerce")
    start = start.where(start.between(1, 2026))
    end = end.where(end.between(1, 2026))
    df["year_opened_midpoint"] = (start + end) / 2
    df = df.drop(columns=["year_opened"])

# Exclude direct identifiers and street-level details.
exclude = ["size", "museum_id", "name_of_museum", "address_line_1", "address_line_2", "postcode"]
X = df.drop(columns=[c for c in exclude if c in df.columns]).dropna(axis=1, how="all")
y = df[target].astype(str)

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = {
    "classifier__n_estimators": [300],
    "classifier__max_depth": [None, 15],
    "classifier__min_samples_leaf": [1, 3],
    "classifier__max_features": ["sqrt"]
}
search = GridSearchCV(pipeline, grid, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)
search.fit(X_train, y_train)

pred = search.predict(X_test)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Best parameters:", search.best_params_)
print("Best CV macro F1:", round(search.best_score_, 4))
print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
print("Test balanced accuracy:", round(balanced_accuracy_score(y_test, pred), 4))
print("Test macro F1:", round(f1_score(y_test, pred, average="macro", zero_division=0), 4))
print("Most-frequent baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 4))
print("\\nClassification report:\\n", classification_report(y_test, pred, zero_division=0))
print("Confusion matrix labels:", sorted(y.unique()))
print(confusion_matrix(y_test, pred, labels=sorted(y.unique())))


Best parameters: {'classifier__max_depth': 15, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 300}
Best CV macro F1: 0.5993
Test accuracy: 0.6874
Test balanced accuracy: 0.553
Test macro F1: 0.5945
Most-frequent baseline accuracy: 0.6229
\nClassification report:\n               precision    recall  f1-score   support

        huge       1.00      0.50      0.67         2
       large       0.42      0.45      0.43       109
      medium       0.50      0.42      0.46       205
       small       0.80      0.84      0.82       522

    accuracy                           0.69       838
   macro avg       0.68      0.55      0.59       838
weighted avg       0.68      0.69      0.68       838

Confusion matrix labels: ['huge', 'large', 'medium', 'small']
[[  1   1   0   0]
 [  0  49  30  30]
 [  0  42  86  77]
 [  0  26  56 440]]
